# Clustering por especimen dentro de genero\n\nMeta (acordada 15-sep, retomada 16-sep): dentro de fotos de planta ya confirmadas, detectar cuales son fotos casi identicas de UN MISMO espécimen fisico (tomas seguidas del mismo arbol/flor), agrupando primero por genero (ya lo predice Pl@ntNet, no depende de tener el arbol de GBIF completo) y DENTRO de cada genero por similitud de imagen.\n\n**Como funciona:**\n1. Este notebook calcula un embedding de imagen (BioCLIP, mismo modelo que `Deteccion_Animal_En_Planta.ipynb`) para cada foto de planta confirmada, y lo guarda en `especimen_embeddings`. Es la parte cara (necesita abrir cada imagen), por eso se cachea aparte y es resumible.\n2. El agrupamiento en si (comparar embeddings por similitud coseno, dentro de cada genero) es barato una vez que los embeddings existen: no se guarda aqui como tabla fija, lo calcula al vuelo la herramienta de revision (`reestructura/clustering_especimen/`), para que nunca quede una propuesta vieja desincronizada de embeddings nuevos.\n3. La decision final (que fotos son del mismo espécimen) la confirma Remi a mano en esa herramienta y se guarda en `especimen_grupos_confirmados`: igual que con toda decision humana en este proyecto, una vez confirmada no la vuelve a tocar ningun proceso automatico.\n\n**Requisito explicito de Remi:** la herramienta de revision debe dejar mover una foto a cualquier grupo aunque su especie/genero predicho por Pl@ntNet no coincida con las demas del grupo, porque una mala prediccion en una sola foto no debe bloquear que el ojo humano la agrupe bien.\n\nSolo cubre fotos disponibles (espejo local u red de ITESO); es resumible, se puede correr en varias sesiones conforme se recupere acceso a mas fotos."

In [ ]:
import concurrent.futures
import json
import sqlite3
from pathlib import Path

import numpy as np
import torch
import open_clip
from PIL import Image

RUTA_BD = Path(\"../../experimentos/bd/campuslab_volumen.db\")
RUTA_CONFIG = Path(\"config_local.json\")
con = sqlite3.connect(RUTA_BD)
print(f\"Base de datos: {RUTA_BD.resolve()} ({'existe' if RUTA_BD.exists() else 'NO ENCONTRADA'})\")

con.execute(\"\"\"
    CREATE TABLE IF NOT EXISTS especimen_embeddings (
        foto_id INTEGER PRIMARY KEY,
        genero TEXT,
        embedding BLOB,
        fecha TEXT DEFAULT CURRENT_TIMESTAMP
    )
\"\"\")
con.commit()

# Mismo mecanismo de red_iteso_disponible() usado en reestructura/*/backend.py: chequeo
# de una sola vez por corrida, con timeout corto, para no colgarse minutos si Y:\\ no
# responde (sin esto, un simple .exists() contra un drive de red caido puede tardar
# muchisimo). Nunca usar \"with ThreadPoolExecutor(...)\", su __exit__ esperaria al hilo
# colgado de todas formas.
_ITESO_DISPONIBLE = None


def red_iteso_disponible():
    global _ITESO_DISPONIBLE
    if _ITESO_DISPONIBLE is None:
        ex = concurrent.futures.ThreadPoolExecutor(max_workers=1)
        futuro = ex.submit(lambda: Path(\"Y:/\").exists())
        try:
            _ITESO_DISPONIBLE = futuro.result(timeout=2)
        except concurrent.futures.TimeoutError:
            _ITESO_DISPONIBLE = False
        ex.shutdown(wait=False)
    return _ITESO_DISPONIBLE


def resolver_ruta_legible(ruta_actual):
    if RUTA_CONFIG.exists():
        config = json.loads(RUTA_CONFIG.read_text(encoding=\"utf-8\"))
        ruta_espejo = config.get(\"ruta_espejo_local_fotos\")
        if ruta_espejo:
            local = Path(ruta_espejo) / Path(ruta_actual).name
            if local.exists():
                return local
    if not red_iteso_disponible():
        return None
    ruta_directa = Path(ruta_actual)
    return ruta_directa if ruta_directa.exists() else None


print(\"Cargando BioCLIP...\")
modelo, _, preprocess = open_clip.create_model_and_transforms(\"hf-hub:imageomics/bioclip\")
modelo.eval()
print(\"Modelo listo.\")

In [ ]:
candidatas_todas = con.execute(\"\"\"
    SELECT f.id, f.ruta_actual, e.nombre_cientifico
    FROM fotos f JOIN especies e ON e.id = f.especie_id
    WHERE f.estado='usable' AND f.ruta_actual NOT LIKE '%.db'
      AND f.id NOT IN (SELECT foto_id FROM especimen_embeddings)
\"\"\").fetchall()
print(f\"Confirmadas sin embedding todavia: {len(candidatas_todas):,}\")

candidatas = []
sin_acceso = 0
for foto_id, ruta, nombre in candidatas_todas:
    ruta_legible = resolver_ruta_legible(ruta)
    if ruta_legible is None:
        sin_acceso += 1
        continue
    genero = nombre.split(\" \")[0]
    candidatas.append((foto_id, ruta_legible, genero))

print(f\"Disponibles ahorita (espejo local o red ITESO): {len(candidatas):,}\")
print(f\"Sin acceso por ahora (quedan pendientes para otra corrida): {sin_acceso:,}\")

In [ ]:
EJECUTAR_REAL = False  # <-- cambialo a True para correr de verdad

if not EJECUTAR_REAL:
    print(\"EJECUTAR_REAL esta en False. No se proceso nada todavia.\")
else:
    procesadas = errores = 0
    for i, (foto_id, ruta_legible, genero) in enumerate(candidatas, 1):
        try:
            img = Image.open(ruta_legible).convert(\"RGB\")
            img_t = preprocess(img).unsqueeze(0)
            with torch.no_grad():
                embedding = modelo.encode_image(img_t)
                embedding /= embedding.norm(dim=-1, keepdim=True)
            vector = embedding.squeeze(0).numpy().astype(np.float32)
            con.execute(
                \"INSERT OR REPLACE INTO especimen_embeddings (foto_id, genero, embedding) VALUES (?,?,?)\",
                (foto_id, genero, vector.tobytes()),
            )
            con.commit()  # commit por foto: la herramienta de revision lee esta tabla en\n            # paralelo, un commit cada N fotos deja el lock de escritura abierto minutos\n            procesadas += 1
        except Exception as e:
            errores += 1
            print(f\"error en foto {foto_id} ({ruta_legible}): {e}\")
        if i % 100 == 0:
            print(f\"  {i}/{len(candidatas)} (ok={procesadas}, errores={errores})\")
    print(f\"\\nListo. procesadas={procesadas}, errores={errores}\")

## Siguiente paso (construido en paralelo)\n\nRevisar las agrupaciones propuestas y confirmarlas a mano en `reestructura/clustering_especimen/` (herramienta web, mismo patron que `revision_web/` y `galeria_animal_en_planta/`). Vuelve a correr este notebook cuando cambie el acceso a fotos (mas red, mas espejo local) para ir completando `especimen_embeddings`; es aditivo y resumible, nunca se sobreescribe una foto ya calculada."